# bench-food
> BBC Good Food dataset — benchmarking ML methods across four tasks.

**Dataset columns confirmed:**
```
id, url, image, name, description, author, rattings, ingredients, steps,
nutrients, times, serves, difficult, vote_count, subcategory, dish_type,
maincategory, recipe_text, embeddings_class (768-d), embeddings_reg (768-d)
```
Embeddings are pre-computed (Gemini `text-embedding-004`).


## Setup


In [ ]:
%rm -rf bench_research_ml_project
!git clone https://github.com/oremaz/bench_research_ml_project
%cd bench_research_ml_project
!pip install -q -r requirements.txt
!pip install -q transformers==4.51.3 bitsandbytes==0.47.0 accelerate optimum auto-gptq
%cd ml_pipeline


In [ ]:
import ast, os, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split as sk_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, accuracy_score

from utils.data import load_csv, prepare_embeddings_data, train_val_test_split, LabelEncoderHelper, filter_meal_types
from utils.metrics import METRIC_REGISTRY
from utils.visualization import plot_confusion_matrix, plot_regression_results
from pipelines_torch.models import MODEL_REGISTRY
from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.base import SimplePredictor
from data_augmentation.augmentations import AUGMENTATION_REGISTRY
from utils.utils import load_model_by_name, RESULTS_DIR

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


## Shared Utilities


### Tabular Feature Extraction

Columns used and their parsing strategy:

| Feature | Source column | Parsing |
| --- | --- | --- |
| `n_ingredients` | `ingredients` | `len(ast.literal_eval(...))` — string repr of a list |
| `n_steps` | `steps` | `len(ast.literal_eval(...))` — string repr of a list |
| `total_time_min` | `times` | `{'Preparation': '15 mins', 'Cooking': 'No Time'}` → sum of numeric parts |
| `recipe_text_len` | `recipe_text` | `len(str(...))` — character count |
| `serves` | `serves` | numeric, filled with median |
| `rattings` | `rattings` | numeric (1–5 integer), filled with median |
| `vote_count` | `vote_count` | log-transformed count, filled with 0 |


In [ ]:
def _safe_list_len(x):
    """Length of a column stored as a stringified Python list."""
    try:
        v = ast.literal_eval(x) if isinstance(x, str) else x
        return len(v) if isinstance(v, (list, tuple)) else 0
    except:
        return 0

def _parse_minutes(s):
    """'15 mins' | 'No Time' | '' → float minutes."""
    if not s: return 0.0
    s = str(s).strip()
    if 'No Time' in s or s == '': return 0.0
    import re
    hrs  = re.search(r'(\d+\.?\d*)\s*hr', s)
    mins = re.search(r'(\d+\.?\d*)\s*min', s)
    total = 0.0
    if hrs:  total += float(hrs.group(1)) * 60
    if mins: total += float(mins.group(1))
    if total == 0.0:
        num = re.search(r'(\d+\.?\d*)', s)
        if num: total = float(num.group(1))
    return total

def _parse_total_time(x):
    """times column: dict-like string → total minutes."""
    try:
        d = ast.literal_eval(x) if isinstance(x, str) else x
        if not isinstance(d, dict): return 0.0
        return sum(_parse_minutes(v) for v in d.values())
    except:
        return 0.0

TABULAR_COLS = ['n_ingredients', 'n_steps', 'total_time_min',
                'recipe_text_len', 'serves', 'rattings', 'log_vote_count']

def add_tabular_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add engineered tabular features to a copy of df."""
    df = df.copy()
    df['n_ingredients']   = df['ingredients'].apply(_safe_list_len)
    df['n_steps']         = df['steps'].apply(_safe_list_len)
    df['total_time_min']  = df['times'].apply(_parse_total_time)
    df['recipe_text_len'] = df['recipe_text'].fillna('').apply(len)
    df['serves']          = pd.to_numeric(df['serves'], errors='coerce')
    df['serves']          = df['serves'].fillna(df['serves'].median())
    df['rattings']        = pd.to_numeric(df['rattings'], errors='coerce')
    df['rattings']        = df['rattings'].fillna(df['rattings'].median())
    df['log_vote_count']  = np.log1p(pd.to_numeric(df['vote_count'], errors='coerce').fillna(0))
    return df

# Sanity check
df_raw = load_csv('recipes_df.csv')
df_raw = add_tabular_features(df_raw)
print(df_raw[TABULAR_COLS].head())
print('\nNaN counts:\n', df_raw[TABULAR_COLS].isna().sum())


### Model Suite Helper


In [ ]:
def get_model_configs(input_dim, num_classes):
    """Four-model suite used for every classification task."""
    return [
        {'name': 'logreg',   'sklearn': True,
         'obj': LogisticRegression(max_iter=1000, random_state=SEED)},
        {'name': 'lightgbm', 'class': MODEL_REGISTRY['lightgbm_classifier'], 'params': {}},
        {'name': 'xgboost',  'class': MODEL_REGISTRY['xgboost_classifier'],  'params': {}},
        {'name': 'mlp',      'class': MODEL_REGISTRY['mlp_classifier'],
         'params': {'input_dim': input_dim, 'num_classes': num_classes}},
    ]

metrics_cls = [METRIC_REGISTRY['f1'], METRIC_REGISTRY['recall'],
               METRIC_REGISTRY['precision'], METRIC_REGISTRY['accuracy']]

def run_logreg(X_tr, y_tr, X_te, y_te, label='logreg'):
    """Train and evaluate sklearn LogisticRegression; returns dict of metrics."""
    sc = StandardScaler()
    lr = LogisticRegression(max_iter=1000, random_state=SEED)
    lr.fit(sc.fit_transform(X_tr), y_tr)
    preds = lr.predict(sc.transform(X_te))
    return {
        'model': label,
        'accuracy': round(accuracy_score(y_te, preds), 4),
        'f1_macro': round(f1_score(y_te, preds, average='macro'), 4),
    }


### LLM Text Augmentation


### Gemini Re-Embedding (for augmented texts)


## Task 1 — Difficulty Classification

Label: `difficult` column (`Easy`, `More effort`, `A challenge` → merged to `More effort`).

Five feature strategies are compared:

| Strategy | Features | Dimension | Key property |
| --- | --- | --- | --- |
| `embeddings_only` | Raw `embeddings_class` | 768 | Semantic baseline |
| `pca_embeddings` | PCA(100) of `embeddings_class` | 100 | Denoised, faster trees |
| `tabular_only` | 7 scaled hand-crafted features | 7 | Structural baseline |
| `pca_plus_tabular` | PCA(100) emb + scaled tabular | 107 | Balanced dims for MLP |
| `tabular_plus_emb` | Scaled tabular + raw emb | 775 | Full info |

**Fix applied:** `StandardScaler` is fitted on each approach's full `X` before training,
so MLP receives features on a uniform scale regardless of the approach.
Tree models (LightGBM, XGBoost) are scale-invariant and unaffected.
PCA is fitted on the full embedding matrix (minor leakage, acceptable for a benchmark).


### 1.1 Data Loading & Feature Preparation


In [ ]:
df = load_csv('recipes_df.csv')
df = df.dropna(subset=['embeddings_class', 'difficult'])
df['difficult'] = df['difficult'].replace({'A challenge': 'More effort'})
print('Difficulty labels:', df['difficult'].value_counts().to_dict())

le_diff = LabelEncoderHelper()
le_diff.fit(df['difficult'])
y_enc = le_diff.transform(df['difficult'])
num_classes_diff = len(np.unique(y_enc))

# Raw embeddings
X_emb, _ = prepare_embeddings_data(df, target_column='difficult', embedding_column='embeddings_class')

# Tabular features
df = add_tabular_features(df)
X_tab_raw = df[TABULAR_COLS].values.astype(np.float32)

# ── Fix 1: StandardScaler on tabular block ────────────────────────────────
# Fitted once here; applied consistently to train and test sets.
tab_scaler = StandardScaler()
X_tab = tab_scaler.fit_transform(X_tab_raw).astype(np.float32)

# ── Fix 2: PCA on embedding block ─────────────────────────────────────────
# n_components=100 keeps ~90% of variance in typical 768-d Gemini embeddings.
# whiten=True gives unit variance per component → fair concatenation with tabular.
PCA_N = 100
pca = PCA(n_components=PCA_N, whiten=True, random_state=SEED)
X_pca_emb = pca.fit_transform(X_emb).astype(np.float32)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.3f} '
      f'({PCA_N} components out of 768)')

# ── StandardScaler on raw embeddings (for MLP consistency) ───────────────
emb_scaler = StandardScaler()
X_emb_scaled = emb_scaler.fit_transform(X_emb).astype(np.float32)

# ── Build the 5 approach matrices ─────────────────────────────────────────
approaches = {
    'embeddings_only':   X_emb_scaled,                         # 768-d (scaled)
    'pca_embeddings':    X_pca_emb,                            # 100-d (whitened PCA)
    'tabular_only':      X_tab,                                # 7-d (scaled)
    'pca_plus_tabular':  np.hstack([X_pca_emb, X_tab]),        # 107-d
    'tabular_plus_emb':  np.hstack([X_tab, X_emb_scaled]),     # 775-d (both scaled)
}

for name, X in approaches.items():
    print(f'  {name:25s}: {X.shape}')


### 1.2 Augmentation Comparison (embeddings approach, MLP)


In [ ]:
# Compare 3 augmentation strategies on the MLP with scaled embeddings
aug_names = ['none', 'borderline_smote', 'kmeans_smote']
augmentations = [AUGMENTATION_REGISTRY[n] for n in aug_names]

BenchmarkRunner(
    model_configs=[{
        'name': 'mlp', 'class': MODEL_REGISTRY['mlp_classifier'],
        'params': {'input_dim': X_emb_scaled.shape[1], 'num_classes': num_classes_diff}
    }],
    augmentations=augmentations,
    metrics=metrics_cls, task_type='classification', device=DEVICE,
    epochs=150, batch_size=32, early_stopping=20,
    use_class_weights=True, dropout=0.3, weight_decay=5e-4,
    learning_rate=1e-4, path_start='difficulty_aug', max_factor=1.0,
).run(X_emb_scaled, y_enc)
print('Augmentation benchmark complete.')


### 1.3 Feature Strategy × Model Benchmark


In [ ]:
# sklearn LogReg — run for all approaches (StandardScaler already applied,
# but run_logreg adds a second scaler which is idempotent on already-scaled data)
logreg_rows = []
for ap_name, X_ap in approaches.items():
    X_tr, X_te, y_tr, y_te = sk_split(X_ap, y_enc, test_size=0.2,
                                       stratify=y_enc, random_state=SEED)
    row = run_logreg(X_tr, y_tr, X_te, y_te, label=f'logreg_{ap_name}')
    row['approach'] = ap_name
    logreg_rows.append(row)
    print(row)

# BenchmarkRunner — LightGBM, XGBoost, MLP
for ap_name, X_ap in approaches.items():
    print(f'\n--- {ap_name} (input_dim={X_ap.shape[1]}) ---')
    runner_cfgs = [
        {'name': 'lightgbm', 'class': MODEL_REGISTRY['lightgbm_classifier'], 'params': {}},
        {'name': 'xgboost',  'class': MODEL_REGISTRY['xgboost_classifier'],  'params': {}},
        {'name': 'mlp',      'class': MODEL_REGISTRY['mlp_classifier'],
         'params': {'input_dim': X_ap.shape[1], 'num_classes': num_classes_diff}},
    ]
    BenchmarkRunner(
        model_configs=runner_cfgs, augmentations=[None],
        task_type='classification', device=DEVICE,
        epochs=150, batch_size=32, use_kfold=False,
        learning_rate=3e-4, path_start=f'difficulty_{ap_name}', random_state=SEED,
    ).run(X_ap, y_enc)


### 1.4 Evaluation on Test Set


In [ ]:
test_df = load_csv('recipes_df_test_bis.csv')
test_df = test_df.dropna(subset=['embeddings_class', 'difficult'])
test_df['difficult'] = test_df['difficult'].replace({'A challenge': 'More effort'})

X_test_emb_raw, _ = prepare_embeddings_data(test_df, target_column='difficult', embedding_column='embeddings_class')
y_test_enc = le_diff.transform(test_df['difficult'])
test_df = add_tabular_features(test_df)
X_test_tab_raw = test_df[TABULAR_COLS].values.astype(np.float32)

# Apply the SAME fitted scalers / PCA from training — no leakage
X_test_tab      = tab_scaler.transform(X_test_tab_raw).astype(np.float32)
X_test_emb      = emb_scaler.transform(X_test_emb_raw).astype(np.float32)
X_test_pca_emb  = pca.transform(X_test_emb_raw).astype(np.float32)

test_X_map = {
    'embeddings_only':   X_test_emb,
    'pca_embeddings':    X_test_pca_emb,
    'tabular_only':      X_test_tab,
    'pca_plus_tabular':  np.hstack([X_test_pca_emb, X_test_tab]),
    'tabular_plus_emb':  np.hstack([X_test_tab, X_test_emb]),
}

all_results = list(logreg_rows)
for ap_name, X_test_ap in test_X_map.items():
    for m_name in ['lightgbm', 'xgboost', 'mlp']:
        try:
            params = ({'input_dim': X_test_ap.shape[1], 'num_classes': num_classes_diff}
                      if m_name == 'mlp' else {})
            cls = MODEL_REGISTRY['mlp_classifier'] if m_name == 'mlp' else MODEL_REGISTRY[f'{m_name}_classifier']
            model = load_model_by_name(cls, m_name, params,
                                       path_start=f'difficulty_{ap_name}', augmentation_name='none')
            probs = SimplePredictor(model=model, task_type='classification', batch_size=64).predict_proba(X_test_ap)
            all_results.append({
                'approach': ap_name, 'model': m_name,
                'accuracy': round(float(METRIC_REGISTRY['accuracy'](y_test_enc, probs)), 4),
                'f1_macro': round(float(METRIC_REGISTRY['f1'](y_test_enc, probs)), 4),
            })
        except Exception as e:
            print(f'  Skipping {m_name} [{ap_name}]: {e}')

res_df = pd.DataFrame(all_results)
res_df.to_csv(os.path.join(RESULTS_DIR, 'difficulty_approach_comparison.csv'), index=False)

pivot = res_df.pivot_table(index='model', columns='approach', values='f1_macro')
print(pivot.round(3))
pivot.plot(kind='bar', figsize=(12, 5), title='Difficulty F1-macro — 5 feature strategies × 4 models')
plt.ylabel('F1-macro'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


### 1.5 Analysis

**What to look for:**
- `pca_embeddings` vs `embeddings_only`: if PCA doesn't hurt, the last 668 dimensions of the Gemini embedding are mostly noise for this task.
- `pca_plus_tabular` vs `pca_embeddings`: if tabular features add signal on top of PCA → structural cues matter beyond semantics.
- `tabular_plus_emb` vs `pca_plus_tabular`: if the full embedding beats PCA in combination with tabular → some information was lost in compression.
- `tabular_only` vs all others: a strong tabular baseline would suggest difficulty is largely structural (step count, time, n_ingredients).
- **MLP on `pca_plus_tabular`** should be the most reliable: balanced dimensions (107-d), both blocks on the same variance scale thanks to PCA whitening + StandardScaler.


## Task 2 — Meal Type Classification


### 2.1 Data Preprocessing


In [ ]:
meal_types = ['Lunch recipes', 'Dinner recipes', 'Breakfast recipes']
meal_df = filter_meal_types(load_csv('recipes_df.csv'), meal_types)
meal_df = meal_df.dropna(subset=['embeddings_class', 'subcategory'])
meal_df = meal_df[meal_df['subcategory'].isin(meal_types)].copy()
print('Meal type distribution:\n', meal_df['subcategory'].value_counts())

X_meal, y_meal = prepare_embeddings_data(
    meal_df, target_column='subcategory', embedding_column='embeddings_class')
le_meal = LabelEncoderHelper()
le_meal.fit(y_meal)
y_meal_enc = le_meal.transform(y_meal)
X_train_meal, X_val_meal, _, y_train_meal, y_val_meal, _ = train_val_test_split(
    X_meal, y_meal_enc, val_size=0.15, test_size=0, stratify=y_meal_enc)
num_classes_meal = len(np.unique(y_train_meal))
print(f'meal dataset: {len(X_meal)} samples | {num_classes_meal} classes')


### 2.2 LLM Data Augmentation — How It Could Work

> **We skip this step in practice** to avoid extra LLM inference and Gemini embedding costs.
> The section below documents the approach for future use.

#### Why it could help here

The meal-type dataset is small (less than a hundred samples per class).
LLM augmentation would synthesise new recipe texts per class, re-embed them with Gemini,
and concatenate them to the training set — effectively multiplying labelled data for free.

#### Sketch of the approach

```python
# 1. For each class, sample N recipe_text values
# 2. Generate paraphrases / informal rewrites with an LLM (Gemini/local Qwen):
augmented_texts = llm_augment_texts(
    sampled_texts,
    instruction='Rewrite this recipe in a more informal tone, keeping all information.'
)

# 3. Re-embed generated texts to stay in the same 768-d Gemini space
aug_embeds = [embed_fn_class(t) for t in augmented_texts]

# 4. Concatenate with the original training embeddings
X_train_aug = np.vstack([X_train_meal, np.array(aug_embeds)])
y_train_aug = np.concatenate([y_train_meal, np.array(augmented_labels)])
```

Skipping for now; `X_train_meal` from 2.1 is used directly.


### 2.3 Training


In [ ]:
X_meal_all = np.vstack([X_train_meal, X_val_meal])
y_meal_all = np.concatenate([y_train_meal, y_val_meal])

meal_runner_cfgs = [
    {'name': 'lightgbm', 'class': MODEL_REGISTRY['lightgbm_classifier'], 'params': {}},
    {'name': 'xgboost',  'class': MODEL_REGISTRY['xgboost_classifier'],  'params': {}},
    {'name': 'mlp',      'class': MODEL_REGISTRY['mlp_classifier'],
     'params': {'input_dim': X_meal_all.shape[1], 'num_classes': num_classes_meal}},
]
BenchmarkRunner(
    model_configs=meal_runner_cfgs, augmentations=[None],
    metrics=metrics_cls, task_type='classification', device=DEVICE,
    epochs=600, batch_size=32, early_stopping=40, use_kfold=False,
    use_class_weights=True, learning_rate=2e-4, weight_decay=1e-4,
    path_start='meal_train', random_state=SEED,
).run(X_meal_all, y_meal_all)
print('Meal type training complete.')


### 2.4 Evaluation


In [ ]:
test_meal_df = load_csv('recipes_df_test.csv')
test_meal_df = test_meal_df[test_meal_df['subcategory'].isin(meal_types)].dropna(
    subset=['embeddings_class', 'subcategory'])
X_test_meal, y_test_meal = prepare_embeddings_data(
    test_meal_df, target_column='subcategory', embedding_column='embeddings_class')
y_test_meal_enc = le_meal.transform(y_test_meal)

X_tr_for_lr, _, y_tr_for_lr, _ = sk_split(X_meal_all, y_meal_all, test_size=0.01,
                                            stratify=y_meal_all, random_state=SEED)
meal_lr_row = run_logreg(X_tr_for_lr, y_tr_for_lr, X_test_meal, y_test_meal_enc, label='logreg')

meal_results = [meal_lr_row]
for m in meal_runner_cfgs:
    try:
        model = load_model_by_name(m['class'], m['name'], m['params'], path_start='meal_train')
        y_pred = SimplePredictor(model=model, task_type='classification', batch_size=64).predict(X_test_meal)
        meal_results.append({'model': m['name'],
            'accuracy': round(accuracy_score(y_test_meal_enc, y_pred), 4),
            'f1_macro': round(f1_score(y_test_meal_enc, y_pred, average='macro'), 4)})
    except Exception as e:
        print(f'Skipping {m["name"]}: {e}')

meal_df_res = pd.DataFrame(meal_results)
meal_df_res.to_csv(os.path.join(RESULTS_DIR, 'meal_type_test_results.csv'), index=False)
meal_df_res.set_index('model').plot(kind='bar', figsize=(10, 4), title='Meal Type Test Metrics')
plt.tight_layout(); plt.show()
meal_df_res


### 2.5 Analysis

- Lunch/Dinner confusion is the main challenge — breakfast recipes are structurally very different.
- LightGBM/XGBoost leverage the high-dimensional Gemini embedding well without overfitting.
- If scores are unsatisfactory, the LLM augmentation sketch in §2.2 provides a clear upgrade path.


## Task 3 — Nutrient Value Prediction (Regression)

Target: numeric values parsed from the `nutrients` column (dict-like string):
`{'kcal': '254', 'fat': '7g', 'saturates': '2g', ...}`.
We predict **kcal, fat, saturates, carbs** simultaneously.
Features: pre-computed `embeddings_reg` (768-d).


### 3.1 Data Preprocessing


In [ ]:
def parse_nutrients(x):
    """nutrients dict → dict of floats, stripping unit suffixes."""
    try:
        d = ast.literal_eval(x) if isinstance(x, str) else x
        if not isinstance(d, dict) or not d: return None
        import re
        return {k: float(re.sub(r'[^\d.]', '', str(v))) for k, v in d.items() if re.search(r'\d', str(v))}
    except:
        return None

nutr_df = load_csv('recipes_df.csv')
nutr_df['nutrients_parsed'] = nutr_df['nutrients'].apply(parse_nutrients)
nutr_df = nutr_df[nutr_df['nutrients_parsed'].apply(lambda x: isinstance(x, dict) and 'kcal' in x)].copy()

TARGET_NUTRIENTS = ['kcal', 'fat', 'saturates', 'carbs']
for col in TARGET_NUTRIENTS:
    nutr_df[col] = nutr_df['nutrients_parsed'].apply(
        lambda d: d.get(col, np.nan) if isinstance(d, dict) else np.nan)
nutr_df = nutr_df.dropna(subset=TARGET_NUTRIENTS + ['embeddings_reg'])

X_nutr, _ = prepare_embeddings_data(nutr_df, target_column=TARGET_NUTRIENTS[0], embedding_column='embeddings_reg')
y_nutr = nutr_df[TARGET_NUTRIENTS].values.astype(np.float32)
output_dim_nutr = y_nutr.shape[1]
print(f'Nutrient dataset: {X_nutr.shape[0]} samples | targets: {TARGET_NUTRIENTS} | output_dim={output_dim_nutr}')
print(pd.DataFrame(y_nutr, columns=TARGET_NUTRIENTS).describe().round(1))


### 3.2 Training


In [ ]:
metrics_reg = [METRIC_REGISTRY['mse'], METRIC_REGISTRY['mae'], METRIC_REGISTRY['r2']]
reg_model_configs = [
    {'name': 'mlp_regressor',      'class': MODEL_REGISTRY['mlp_regressor'],
     'params': {'input_dim': X_nutr.shape[1], 'output_dim': output_dim_nutr}},
    {'name': 'xgboost_regressor',  'class': MODEL_REGISTRY['xgboost_regressor'],  'params': {}},
    {'name': 'lightgbm_regressor', 'class': MODEL_REGISTRY['lightgbm_regressor'], 'params': {}},
]
BenchmarkRunner(
    model_configs=reg_model_configs, augmentations=[None],
    metrics=metrics_reg, task_type='regression', device=DEVICE,
    epochs=400, batch_size=32, early_stopping=20, use_kfold=False,
    learning_rate=1e-4, weight_decay=1e-4,
    path_start='nutrient_train', random_state=SEED,
).run(X_nutr, y_nutr)
print('Nutrient regression training complete.')


### 3.3 Evaluation


In [ ]:
test_nutr_df = load_csv('recipes_df_test.csv')
test_nutr_df['nutrients_parsed'] = test_nutr_df['nutrients'].apply(parse_nutrients)
test_nutr_df = test_nutr_df[test_nutr_df['nutrients_parsed'].apply(
    lambda x: isinstance(x, dict) and 'kcal' in x)].copy()
for col in TARGET_NUTRIENTS:
    test_nutr_df[col] = test_nutr_df['nutrients_parsed'].apply(
        lambda d: d.get(col, np.nan) if isinstance(d, dict) else np.nan)
test_nutr_df = test_nutr_df.dropna(subset=TARGET_NUTRIENTS + ['embeddings_reg'])
X_test_nutr, _ = prepare_embeddings_data(test_nutr_df, target_column=TARGET_NUTRIENTS[0], embedding_column='embeddings_reg')
y_test_nutr = test_nutr_df[TARGET_NUTRIENTS].values.astype(np.float32)

nutr_results = []
for m in reg_model_configs:
    try:
        model = load_model_by_name(m['class'], m['name'], m['params'],
                                   path_start='nutrient_train', augmentation_name='none')
        preds = SimplePredictor(model=model, task_type='regression', batch_size=64).predict(X_test_nutr)
        nutr_results.append({'model': m['name'],
            **{str(metric): round(float(metric(y_test_nutr, preds)), 4) for metric in metrics_reg}})
    except Exception as e:
        print(f'Skipping {m["name"]}: {e}')

nutr_res_df = pd.DataFrame(nutr_results)
plot_regression_results(nutr_res_df)
nutr_res_df


### 3.4 Analysis

- `kcal` and `fat` are the most predictable nutrients from recipe text; `saturates` and `carbs` are noisier.
- LightGBM/XGBoost can also use tabular features (n_ingredients, n_steps) if added to the embedding.


## Task 4 — Total Time Classification (Binned)

Total time = `Preparation + Cooking` parsed from `times` column, then binned:

| Bin | Range |
| --- | --- |
| 0 | < 15 min |
| 1 | 15–30 min |
| 2 | 30–60 min |
| 3 | ≥ 60 min |


### 4.1 Data Preprocessing


In [ ]:
def time_bin(t):
    if t < 15:  return 0
    elif t < 30: return 1
    elif t < 60: return 2
    else:        return 3

time_df = load_csv('recipes_df.csv')
time_df = add_tabular_features(time_df)
time_df['total_time_bin'] = time_df['total_time_min'].apply(time_bin)
valid_time_df = time_df[time_df['total_time_min'] > 0].copy()
valid_time_df = valid_time_df.dropna(subset=['embeddings_class'])

X_timec, _ = prepare_embeddings_data(
    valid_time_df, target_column='total_time_bin', embedding_column='embeddings_class')
y_timec = valid_time_df['total_time_bin'].values.astype(int)
num_classes_time = len(np.unique(y_timec))
print('Time bin distribution:', dict(zip(*np.unique(y_timec, return_counts=True))))


### 4.2 Training


In [ ]:
time_runner_cfgs = [
    {'name': 'lightgbm', 'class': MODEL_REGISTRY['lightgbm_classifier'], 'params': {}},
    {'name': 'xgboost',  'class': MODEL_REGISTRY['xgboost_classifier'],  'params': {}},
    {'name': 'mlp',      'class': MODEL_REGISTRY['mlp_classifier'],
     'params': {'input_dim': X_timec.shape[1], 'num_classes': num_classes_time}},
]
BenchmarkRunner(
    model_configs=time_runner_cfgs, augmentations=[None],
    metrics=metrics_cls, task_type='classification', device=DEVICE,
    epochs=150, batch_size=32, use_kfold=False, use_class_weights=True,
    learning_rate=1e-4, weight_decay=5e-4,
    path_start='total_time_train', random_state=SEED,
).run(X_timec, y_timec)
print('Total time classification training complete.')


### 4.3 Evaluation


In [ ]:
test_time_df = load_csv('recipes_df_test_bis.csv')
test_time_df = add_tabular_features(test_time_df)
test_time_df['total_time_bin'] = test_time_df['total_time_min'].apply(time_bin)
valid_test = test_time_df[test_time_df['total_time_min'] > 0].dropna(subset=['embeddings_class'])

X_test_time, _ = prepare_embeddings_data(
    valid_test, target_column='total_time_bin', embedding_column='embeddings_class')
y_test_time = valid_test['total_time_bin'].values.astype(int)

X_tr_t, _, y_tr_t, _ = sk_split(X_timec, y_timec, test_size=0.01, stratify=y_timec, random_state=SEED)
time_results = [run_logreg(X_tr_t, y_tr_t, X_test_time, y_test_time, label='logreg')]

for m in time_runner_cfgs:
    try:
        model = load_model_by_name(m['class'], m['name'], m['params'],
                                   path_start='total_time_train', augmentation_name='none')
        probs = SimplePredictor(model=model, task_type='classification', batch_size=64).predict_proba(X_test_time)
        time_results.append({'model': m['name'],
            'accuracy': round(float(METRIC_REGISTRY['accuracy'](y_test_time, probs)), 4),
            'f1_macro': round(float(METRIC_REGISTRY['f1'](y_test_time, probs)), 4)})
    except Exception as e:
        print(f'Skipping {m["name"]}: {e}')

time_res_df = pd.DataFrame(time_results).sort_values('f1_macro', ascending=False)
time_res_df.to_csv(os.path.join(RESULTS_DIR, 'total_time_test_results.csv'), index=False)
time_res_df


### 4.4 Analysis

- `total_time_min` used for binning is parsed from the `times` column — the same feature used as a tabular predictor in Task 1.
  This means bins 2 and 3 should be well-captured by the tabular approach, while the embedding may help with edge cases.
- The class imbalance (bin 1 and 2 dominate) is handled via `use_class_weights=True`.
